# ПР-8. Градиентный бустинг: прогноз оттока абонентов

Данные: `o_churn_synthetic.csv`, 3000 абонентов, целевой признак `churn`. Датасет синтетический, это не данные реального оператора.
Разбиение сделано по тому же рецепту, что в ПР-7: 80/20, стратификация по `churn`, `random_state=42`. В ПР-7 я работал с другими данными (Titanic), поэтому случайный лес здесь обучен заново на этих данных.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
import sklearn
import xgboost

DATA_PATH = "../data/o_churn_synthetic.csv"

df = pd.read_csv(DATA_PATH)
X = df.drop(columns=["churn", "subscriber_code"])
y = df["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

def make_prep():
    return ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])],
        remainder="passthrough")

print("train:", X_train.shape, " test:", X_test.shape)
print("xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__)


train: (2400, 11)  test: (600, 11)
xgboost 3.4.1 | scikit-learn 1.9.1


In [2]:
print("Пропуски:", df.isna().sum().sum())
print(y.value_counts())
print("Регионы:", df["region"].nunique(), " тарифы:", df["tariff"].nunique())

Пропуски: 0
churn
0    1886
1    1114
Name: count, dtype: int64
Регионы: 9  тарифы: 4


## Веха 8.1. Бустинг по шаблону

Функция `fit_and_report` обучает модель и возвращает F1 на train и test, ROC-AUC на test и время обучения.
Все модели дальше проверяются одной и той же функцией, поэтому сравнение честное.

In [3]:
import time
from sklearn.metrics import f1_score, roc_auc_score

def fit_and_report(name, pipe):
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    sec = time.perf_counter() - t0
    return {"Модель": name,
            "F1 train": f1_score(y_train, pipe.predict(X_train)),
            "F1 test": f1_score(y_test, pipe.predict(X_test)),
            "ROC-AUC test": roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1]),
            "Время, с": sec}

In [4]:
from xgboost import XGBClassifier

xgb = make_pipeline(make_prep(), XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42))
res_xgb = fit_and_report("XGBoost, шаблон", xgb)
pd.DataFrame([res_xgb]).set_index("Модель").round(3)

,F1 train,F1 test,ROC-AUC test,"Время, с"
Модель,,,,
"XGBoost, шаблон",0.789,0.574,0.779,0.487


## Веха 8.2. Случайный лес из ПР-7 на тех же данных

Обучаю лес из 100 деревьев, остальные параметры оставляю по умолчанию. В ПР-7 я подбирал параметры леса под Titanic (`max_depth=10`, `min_samples_leaf=4`), здесь беру базовый лес и проверяю его той же функцией `fit_and_report`.

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = make_pipeline(make_prep(), RandomForestClassifier(
    n_estimators=100, random_state=42))
res_rf = fit_and_report("Случайный лес, 100", rf)
pd.DataFrame([res_rf, res_xgb]).set_index("Модель").round(3)

,F1 train,F1 test,ROC-AUC test,"Время, с"
Модель,,,,
"Случайный лес, 100",1.000,0.556,0.752,0.385
"XGBoost, шаблон",0.789,0.574,0.779,0.487


### Сравнение моделей

В таблице выше числа посчитаны кодом по текущему запуску.

**Какая модель лучше по test?** XGBoost: у него выше и F1, и ROC-AUC, чем у леса.

**А по разрыву между train и test?** Тоже XGBoost. У леса F1 train = 1.000, а F1 test около 0.55, разрыв около 0.45. У XGBoost разрыв 0.215 (0.789 − 0.574).
F1 train = 1.000 у леса значит, что он запомнил обучающую выборку, как одно дерево в ПР-7.
Глубокие деревья леса подгоняются под каждого абонента в train. Бустинг строит неглубокие деревья (`max_depth=4`) с маленьким шагом, поэтому переобучается меньше.

## Веха 8.3. Чем бустинг отличается от леса

В случайном лесу деревья строятся независимо друг от друга: каждое учится на своей случайной подвыборке, а ответ получается голосованием.
В бустинге деревья строятся по очереди, и каждое новое дерево учится исправлять ошибки уже построенных: оно обучается на том, что предыдущие деревья предсказали неправильно, а `learning_rate` уменьшает его вклад.
Поэтому лишние деревья лесу почти не вредят, а бустинг от них может переобучиться: после какого-то числа деревьев он начинает подгоняться под шум в train.
Значит, число деревьев в бустинге нужно подбирать, например ранней остановкой по валидационной выборке.

## Веха 8.4. Валидационная выборка

Для ранней остановки нужна отдельная выборка, на которой видно, когда бустинг начинает переобучаться.
Беру её из train (20%), test не трогаю: он остаётся для финальной оценки.
Препроцессор обучаю только на `X_tr`, а `X_val` и `X_test` он только преобразует.

In [6]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
p = make_prep().fit(X_tr)
Xtr, Xval = p.transform(X_tr), p.transform(X_val)
print("строк:", len(X_tr), len(X_val), len(X_test),
      " признаков после one-hot:", Xtr.shape[1])

строк: 1920 480 600  признаков после one-hot: 22


## Веха 8.5. Ранняя остановка

Ставлю заведомо много деревьев (2000), а после каждого дерева считаю logloss на валидации.
Если 50 деревьев подряд logloss не улучшают, обучение останавливается, и модель берёт лучшую итерацию.

In [7]:
es = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4,
                   eval_metric="logloss", early_stopping_rounds=50,
                   random_state=42)
es.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)
print("лучшая итерация:", es.best_iteration,
      " logloss на val:", round(es.best_score, 4))

лучшая итерация: 78  logloss на val: 0.5405


**Вывод.** Лучший logloss на валидации (0.5405) получился на итерации 78, то есть хватает 79 деревьев (счёт итераций идёт с нуля).
В шаблоне 8.1 было 300 деревьев, почти в 4 раза больше нужного.
После 79-го дерева ещё 50 деревьев не улучшили logloss, поэтому обучение остановилось на 129-м.

## Веха 8.6. Перебор learning_rate и max_depth

Перебираю 4 значения шага и 3 глубины, всего 12 комбинаций. Число деревьев в каждой подбирает ранняя остановка, качество сравниваю по ROC-AUC на валидации. Test не трогаю.

In [8]:
for lr in [0.3, 0.1, 0.05, 0.01]:
    for depth in [3, 4, 6]:
        m = XGBClassifier(n_estimators=3000, learning_rate=lr,
                          max_depth=depth, eval_metric="logloss",
                          early_stopping_rounds=50, random_state=42)
        m.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)
        auc = roc_auc_score(y_val, m.predict_proba(Xval)[:, 1])
        n = m.best_iteration + 1
        print(f"lr={lr:<5} depth={depth}  деревьев={n:<4} AUC val={auc:.3f}")

lr=0.3   depth=3  деревьев=19   AUC val=0.770
lr=0.3   depth=4  деревьев=11   AUC val=0.773


lr=0.3   depth=6  деревьев=8    AUC val=0.757
lr=0.1   depth=3  деревьев=72   AUC val=0.778


lr=0.1   depth=4  деревьев=47   AUC val=0.780
lr=0.1   depth=6  деревьев=36   AUC val=0.766


lr=0.05  depth=3  деревьев=127  AUC val=0.783


lr=0.05  depth=4  деревьев=79   AUC val=0.776
lr=0.05  depth=6  деревьев=85   AUC val=0.768


lr=0.01  depth=3  деревьев=652  AUC val=0.780


lr=0.01  depth=4  деревьев=511  AUC val=0.775


lr=0.01  depth=6  деревьев=307  AUC val=0.763


**Вывод.** Чем меньше шаг, тем больше деревьев нужно: при depth=3 от 19 деревьев (lr=0.3) до 652 (lr=0.01).
Качество от шага зависит слабо: при depth=3 AUC растёт с 0.770 (lr=0.3) до 0.783 (lr=0.05) и дальше не растёт (0.780 при lr=0.01). Снижать шаг ниже 0.05 смысла нет: деревьев нужно примерно в 5 раз больше, а качество то же.
AUC на валидации у всех 12 комбинаций от 0.757 до 0.783, разница в пределах шума.
Глубина 6 везде хуже, глубокие деревья здесь переобучаются быстрее.
Выбираю lr=0.05, depth=3, 127 деревьев: лучший AUC val и при этом не слишком много деревьев.

## Веха 8.7. Итоговое сравнение трёх моделей

Логистическая регрессия, случайный лес из ПР-7 и XGBoost с параметрами из 8.6 на одном и том же разбиении.
Для логрегрессии числовые признаки масштабирую (`StandardScaler`), деревьям это не нужно.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaled = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["region", "tariff"])],
    remainder=StandardScaler())
models = {
    "Логрегрессия": make_pipeline(scaled, LogisticRegression(max_iter=1000)),
    "Случайный лес": make_pipeline(make_prep(), RandomForestClassifier(
        n_estimators=100, random_state=42)),
    "XGBoost": make_pipeline(make_prep(), XGBClassifier(
        n_estimators=127, learning_rate=0.05, max_depth=3, random_state=42)),
}
rows = [fit_and_report(name, pipe) for name, pipe in models.items()]
summary = pd.DataFrame(rows).set_index("Модель").round(3)
summary


,F1 train,F1 test,ROC-AUC test,"Время, с"
Модель,,,,
Логрегрессия,0.594,0.611,0.806,0.024
Случайный лес,1.000,0.556,0.752,0.367
XGBoost,0.645,0.586,0.792,0.051


### Итог

Итоговая таблица с метриками на train и test и временем обучения выведена кодом выше.

На этих данных лучше всех простая логистическая регрессия: у неё самые высокие F1 (0.611) и ROC-AUC (0.806) на test, и она почти не переобучается (F1 на train даже ниже, чем на test).
Она же быстрее всех: обучение занимает сотые доли секунды. Дольше всех учится случайный лес из-за 100 глубоких деревьев.
Подбор параметров немного помог XGBoost: ROC-AUC вырос с 0.779 (шаблон 8.1) до 0.792, разрыв train/test по F1 уменьшился с 0.215 до 0.059. Это небольшой прирост на тесте из 600 абонентов, так что делать из него сильный вывод не стоит.
Бустинг не обязан быть лучше. Если зависимость оттока от признаков близка к линейной, сложная модель ничего не выигрывает и только подгоняется под шум.